In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2026 NVIDIA Corporation
Built on Thu_Mar_19_22:28:55_Pacific_Daylight_Time_2026
Cuda compilation tools, release 13.2, V13.2.78
Build cuda_13.2.r13.2/compiler.37668154_0


In [2]:
%%writefile vector_add.cu

#include <stdio.h>
#include <stdlib.h>

// CUDA kernel for vector addition
__global__ void vectorAdd(int* a, int* b, int* c, int size)
{
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < size) {
        c[tid] = a[tid] + b[tid];
    }
}

int main()
{
    int size = 100;

    int *a, *b, *c;
    int *dev_a, *dev_b, *dev_c;

    // Allocate host memory
    a = (int*)malloc(size * sizeof(int));
    b = (int*)malloc(size * sizeof(int));
    c = (int*)malloc(size * sizeof(int));

    // Initialize vectors
    for (int i = 0; i < size; i++) {
        a[i] = i;
        b[i] = 2 * i;
    }

    // Allocate device memory
    cudaMalloc((void**)&dev_a, size * sizeof(int));
    cudaMalloc((void**)&dev_b, size * sizeof(int));
    cudaMalloc((void**)&dev_c, size * sizeof(int));

    // Copy data to GPU
    cudaMemcpy(dev_a, a, size * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(dev_b, b, size * sizeof(int), cudaMemcpyHostToDevice);

    // Launch kernel
    int blockSize = 256;
    int gridSize = (size + blockSize - 1) / blockSize;

    vectorAdd<<<gridSize, blockSize>>>(dev_a, dev_b, dev_c, size);

    // Copy result back
    cudaMemcpy(c, dev_c, size * sizeof(int), cudaMemcpyDeviceToHost);

    // Print output
    for (int i = 0; i < size; i++) {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
    }

    // Free memory
    cudaFree(dev_a);
    cudaFree(dev_b);
    cudaFree(dev_c);

    free(a);
    free(b);
    free(c);

    return 0;
}

Writing vector_add.cu


In [3]:
!nvcc vector_add.cu -o vector_add

nvcc fatal   : Cannot find compiler 'cl.exe' in PATH


In [4]:
!./vector_add

'.' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
%%writefile matrix_mul.cu

#include <stdio.h>
#include <stdlib.h>

// CUDA kernel for matrix multiplication
__global__ void matrixMul(int* a, int* b, int* c,
                          int rowsA, int colsA, int colsB)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    int sum = 0;

    if (row < rowsA && col < colsB) {

        for (int i = 0; i < colsA; i++) {
            sum += a[row * colsA + i] *
                   b[i * colsB + col];
        }

        c[row * colsB + col] = sum;
    }
}

int main()
{
    int rowsA = 10;
    int colsA = 10;
    int rowsB = colsA;
    int colsB = 10;

    int *a, *b, *c;
    int *dev_a, *dev_b, *dev_c;

    // Allocate host memory
    a = (int*)malloc(rowsA * colsA * sizeof(int));
    b = (int*)malloc(rowsB * colsB * sizeof(int));
    c = (int*)malloc(rowsA * colsB * sizeof(int));

    // Initialize matrices
    for (int i = 0; i < rowsA * colsA; i++) {
        a[i] = i;
    }

    for (int i = 0; i < rowsB * colsB; i++) {
        b[i] = 2 * i;
    }

    // Allocate device memory
    cudaMalloc((void**)&dev_a, rowsA * colsA * sizeof(int));
    cudaMalloc((void**)&dev_b, rowsB * colsB * sizeof(int));
    cudaMalloc((void**)&dev_c, rowsA * colsB * sizeof(int));

    // Copy data to GPU
    cudaMemcpy(dev_a, a, rowsA * colsA * sizeof(int),
               cudaMemcpyHostToDevice);

    cudaMemcpy(dev_b, b, rowsB * colsB * sizeof(int),
               cudaMemcpyHostToDevice);

    // Grid and block dimensions
    dim3 blockSize(16,16);

    dim3 gridSize((colsB + blockSize.x - 1) / blockSize.x,
                  (rowsA + blockSize.y - 1) / blockSize.y);

    // Launch kernel
    matrixMul<<<gridSize, blockSize>>>(dev_a, dev_b, dev_c,
                                       rowsA, colsA, colsB);

    // Copy result back
    cudaMemcpy(c, dev_c,
               rowsA * colsB * sizeof(int),
               cudaMemcpyDeviceToHost);

    // Print result
    printf("Result Matrix:\n");

    for (int i = 0; i < rowsA; i++) {

        for (int j = 0; j < colsB; j++) {
            printf("%d ", c[i * colsB + j]);
        }

        printf("\n");
    }

    // Free memory
    cudaFree(dev_a);
    cudaFree(dev_b);
    cudaFree(dev_c);

    free(a);
    free(b);
    free(c);

    return 0;
}

Writing matrix_mul.cu


In [6]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc fatal   : Cannot find compiler 'cl.exe' in PATH


In [7]:
!./matrix_mul

'.' is not recognized as an internal or external command,
operable program or batch file.
